# SBI on Simulated vs Experimental Summary Features

This notebook trains an amortized posterior on simulated parameter-set summaries and evaluates it on experimental summary observations. It supports three observation levels only:
- participant x condition
- mean per condition
- grand mean across participant-condition rows


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import numpy as np
import pandas as pd

from sbi_sim_exp_inference import (
    build_selection_templates,
    compute_network_posterior_modes,
    compute_posterior_modes,
    create_run_output_dir,
    discover_available_feature_groups,
    discover_available_parameter_groups,
    get_feature_scaling_metadata,
    load_summary_table,
    load_trained_inference_bundle,
    plot_feature_parameter_relevance_grid,
    plot_feature_parameter_correlation_heatmaps,
    plot_conditional_posterior_slices,
    plot_internal_validation_accuracy,
    plot_internal_validation_calibration,
    plot_posterior_condition_diagnostics,
    plot_posterior_pairplots,
    plot_posterior_pca_columns,
    prepare_inference_dataset,
    prepare_observation_dataset_for_inference,
    run_internal_posterior_validation,
    save_inference_outputs,
    save_internal_validation_outputs,
    sample_ensemble_posteriors_for_observations,
    sample_posteriors_for_observations,
    summarize_posterior_samples,
    train_amortized_posterior,
    train_posterior_ensemble,
)


In [ ]:
# INPUT / OUTPUT PATHS
# Simulated table: one row = one simulated parameter-set batch summary.
# Experimental table: one row = one participant x session x condition batch summary.
# Use PKL by default because it preserves native objects better than CSV.
SIMULATED_TABLE_PATH = Path("sim_batches_GONOGO/parameter_set_summaries/parameter_set_feature_summary.pkl")
EXPERIMENTAL_TABLE_PATH = Path("EXP_DATA_ANALYZED_HDF5/_batch_summaries/experimental_batch_feature_summary.pkl")
OUTPUT_ROOT_DIR = Path("sbi_output_GONOGO")

# RUN MODE
# 'train' = prepare dataset from notebook selections, train a new SBI network, then sample posteriors.
# 'reload' = load an existing trained SBI bundle, ignore notebook parameter/feature selections,
#            prepare observations from EXPERIMENTAL_TABLE_PATH using the saved preprocessing,
#            then sample posteriors without retraining.
RUN_MODE = "train"
RELOAD_BUNDLE_PATH = OUTPUT_ROOT_DIR / "SBI_RUN_normalize_to_exp_FEATURES_all_windows" / "trained_inference_bundle.pkl"

# OBSERVATION LEVEL
# 'participant_condition' = one posterior per participant x session x condition row.
# 'condition_mean' = one posterior per condition mean row.
# 'grand_mean' = one posterior for the mean across all participant-condition rows.
OBSERVATION_LEVEL = "condition_mean"  # "participant_condition", "condition_mean", "grand_mean"

# FEATURE NORMALIZATION
# 'simulated' = z-score features using retained simulated training rows.
# 'experimental' = z-score features using experimental observations aggregated at FEATURE_NORMALIZATION_EXPERIMENTAL_LEVEL.
FEATURE_NORMALIZATION_SOURCE = "experimental"  # "simulated", "experimental"
FEATURE_NORMALIZATION_EXPERIMENTAL_LEVEL = "participant_condition"

# SBI / SAMPLING SETTINGS
DENSITY_ESTIMATOR = "maf"
TRAIN_DEVICE = "cpu"
SHOW_SBI_PROGRESS_BARS = True
N_TRAINED_NETWORKS = 5  # Equal-weight ensemble size; 1 preserves the historical single-network workflow.
ENSEMBLE_BASE_SEED = 12345  # Deterministically generates one training seed per network.
POSTERIOR_NUM_SAMPLES = 10_000 # 10_000  # Total pooled posterior samples across all networks.
POSTERIOR_SAMPLE_SEED = 12345

# Conservative defaults for a first pass. Increase later if needed.
SNPE_TRAIN_KWARGS = {
    "training_batch_size": 128, # 200,
    "learning_rate": 2e-4, # 5e-4,
    "validation_fraction": 0.1,
    "stop_after_epochs": 20,
    "show_train_summary": True,
}

# OPTIONAL MANUAL PARAMETER BOUND OVERRIDES
# Use exported parameter names as keys, e.g. "SIM_PARAM_GENERAL_tau_size_um": (0.0, 10.0)
# Leave empty to use bounds resolved automatically from *_priors_batch0.pkl.
PARAMETER_BOUNDS_OVERRIDES = {
    # "SIM_PARAM_GENERAL_tau_size_um": (0.0, 10.0),
}

# OUTPUT OPTIONS
SAVE_POSTERIOR_SAMPLES_PKL = True
SAVE_TRAINED_BUNDLE_PKL = True
SAVE_TRAINING_DIAGNOSTICS = True

# POSTERIOR MODE SETTINGS
POSTERIOR_MODE_KNN_NEIGHBORS = 25
POSTERIOR_MODE_MIN_NEIGHBORS = 5

# PAIRPLOT SETTINGS
# For participant_condition, leave PAIRPLOT_LABELS = None to plot only the first available label.
# For condition_mean, None plots all condition means. For grand_mean, None plots the single posterior.
PAIRPLOT_LABELS = None
PARAMETER_COLORS = {
    "SIM_PARAM_GENERAL_tau_size_um": "#00CFAD",
    # Baseline Input
    "SIM_PARAM_BASELINE_INPUT_common_input_weight_sd": "#FF7300",
    "SIM_PARAM_BASELINE_INPUT_excitatory_input_baseline": "#FF7300",
    "SIM_PARAM_BASELINE_INPUT_lf_target_sd": "#ff3300",
    "SIM_PARAM_BASELINE_INPUT_alpha_target_sd_final": "#ffa600",
    "SIM_PARAM_BASELINE_INPUT_beta_target_sd_final": "#008cff",
    "SIM_PARAM_BASELINE_INPUT_independent_input_power": "#FF75DC",
    "SIM_PARAM_BASELINE_INPUT_independent_input_weight_sd": "#FF75DC",
    # Burst Input
    # "SIM_PARAM_BURST_INPUT_input_burst_sigma_ms": "#999999",
    "SIM_PARAM_BURST_INPUT_lf_burst_peak_nA": "#ff3300",
    "SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA": "#ffa600",
    "SIM_PARAM_BURST_INPUT_beta_burst_peak_nA": "#008cff",
    # Trend Input
    "SIM_PARAM_TREND_INPUT_lf_trend_slope_nA_per_s": "#ff3300",
    "SIM_PARAM_TREND_INPUT_alpha_trend_slope_nA_per_s": "#ffa600",
    "SIM_PARAM_TREND_INPUT_beta_trend_slope_nA_per_s": "#008cff",
}
FEATURES_COLORS = {
    "cst_low_frequency": "#ff3300",
    "cst_alpha": "#ffa600",
    "cst_beta": "#008cff",
    "sync_coincidence": "#852680",
    "LF": "#ff3300",
    "ALPHA": "#ffa600",
    "BETA": "#008cff",
    "SYNC": "#852680",
}
EXPERIMENTAL_COLORS = {
    "HAND_GO": "#ffbb00",
    "HAND_NOGO": "#ff6600",
    "TA_NOGO": "#ff0080",
}
PAIRPLOT_STYLE = {
    "figure_dpi": 140,
    "cell_size": 2.8,
    "scatter_alpha": 0.02,
    "scatter_size": 25,
    "diagonal_kind": "kde",  # "kde"
    "diagonal_kde_points": 256,
    "diagonal_kde_bandwidth_scale": 1.0,
    "hist_alpha": 0.5,
    "hist_bins": 20,
    "label_all_axes": True,
    "color": "#007BFF",
    "parameter_colors": PARAMETER_COLORS,
    "joint_color_blend_space": "lab",  # "lab", "rgb", or "hsv"
    "show_pair_correlations": True,
    "pair_correlation_fontsize": 8.0,
    "pair_correlation_loc": "upper left",
    "pair_correlation_bbox_alpha": 0.78,
    "show_mode_marker": True,
    "mode_marker_color": "#000000",
    "mode_marker_size": 90.0,
    "mode_marker_alpha": 1.0,
    "mode_marker_symbol": "X",
    "show_mode_diagonal_line": True,
    "mode_diagonal_linewidth": 1.5,
    "mode_diagonal_linestyle": "--",
    "network_density_color": None,  # None = use the same color as the ensemble marginal density.
    "network_density_alpha": 0.5,
    "network_density_linewidth": 1.5,
    "normalize_diagonal_density_ylim": True,
}

POSTERIOR_CONDITION_DIAGNOSTICS_ENABLED = True
POSTERIOR_CONDITION_DIAGNOSTICS_LABELS = None
POSTERIOR_CONDITION_DIAGNOSTICS_FILENAME = "posterior_condition_diagnostics.png"
POSTERIOR_CONDITION_DIAGNOSTICS_STYLE = {
    "figure_dpi": 200,
    "cell_width": 4.0,
    "density_height": 3.5,
    "heatmap_height": 3.5,
    "kde_points": 256,
    "kde_bandwidth_scale": 1.0,
    "density_linewidth": 1.8,
    "density_fill_alpha": 0.16,
    "density_line_alpha": 0.95,
    "ridge_density_scale": 1.0,
    "p_gt_zero_text_fontsize": 7.5,
    "show_zero_reference_line": True,
    "p_delta_cmap": "coolwarm",
    "p_delta_text_fontsize": 8.0,
    "tick_labelsize": 8.0,
    "parameter_label_fontsize": 9.0,
    "title_fontsize": 12.0,
    "legend_fontsize": 8.0,
    "show_density_legend": True,
    "title_y": 0.99,
    "legend_y": 0.93,
    "figure_left": 0.06,
    "figure_right": 0.90,
    "figure_bottom": 0.16,
    "figure_top": 0.80,
    "subplot_wspace": 0.5,
    "subplot_hspace": 0.78,
    "colorbar_shrink": 0.78,
    "colorbar_pad": 0.035,
    "condition_colors": EXPERIMENTAL_COLORS,
    "condition_cmap": "tab10",
    "parameter_colors": PARAMETER_COLORS,
}

POSTERIOR_PCA_ENABLED = True
POSTERIOR_PCA_LABELS = None
POSTERIOR_PCA_FILENAME_PREFIX = "posterior_pca"
POSTERIOR_PCA_N_COMPONENTS_DISPLAY = None
POSTERIOR_PCA_STYLE = {
    "figure_dpi": 150,
    "left_column_width": 7.2,
    "right_column_width": 5.8,
    "base_height": 6.6,
    "per_pc_height": 1.6,
    "scatter_alpha": 0.02,
    "scatter_size": 20.0,
    "scatter_color": "#2BB1FF",
    "parameter_colors": PARAMETER_COLORS,
    "standardize_parameters": True,
    "show_mode_marker": True,
    "mode_marker_color": "#000000",
    "mode_marker_size": 100.0,
    "mode_marker_alpha": 1.0,
    "mode_marker_symbol": "X",
    "loading_bar_edgecolor": "white",
    "loading_bar_linewidth": 0.5,
    "loading_tick_labelsize": 8.0,
    "loading_title_fontsize": 10.0,
    "loading_text_fontsize": 8.5,
    "loading_text_loc": "upper right",
    "share_loading_ylims": True,
    "projection_marker_size": 10.0,
    "projection_alpha": 0.02,
    "projection_color": "#2BB1FF",
    "projection_plane_alpha": 0.06,
    "projection_grid_alpha": 0.18,
    "projection_edge_color": "#555555",
    "projection_edge_lw": 1.4,
    "projection_edge_alpha": 0.9,
    "projection_elev": 22.0,
    "projection_azim": -54.0,
    "projection_padding_fraction": 0.08,
    "projection_label_fontsize": 9.0,
    "projection_tick_labelsize": 8.0,
}


CONDITIONAL_SLICE_EXPLORER_ENABLED = True
CONDITIONAL_SLICE_LABELS = None
CONDITIONAL_SLICE_CONSTRAINED_PARAMS = ["SIM_PARAM_TREND_INPUT_alpha_trend_slope_nA_per_s"]
CONDITIONAL_SLICE_PERCENTILES = [5, 95] # [10, 50, 90]
CONDITIONAL_SLICE_INCLUDE_MODE = False # True
CONDITIONAL_SLICE_REFERENCE_SAMPLES = None
CONDITIONAL_SLICE_N_CONDITIONAL_SAMPLES = 600  # Total pooled conditional MCMC samples across all networks.
CONDITIONAL_SLICE_RANDOM_SEED = 12345
CONDITIONAL_SLICE_MISMATCH_WARN_THRESHOLD = 8.0
CONDITIONAL_SLICE_MCMC_STYLE = {
    "conditional_mcmc_method": "slice_np_vectorized",
    "conditional_warmup_steps": 200,
    "conditional_num_chains": 8,
    "conditional_thin": 1,
    "conditional_num_workers": 16, # 1,
    "conditional_mp_context": "spawn",
    "conditional_init_strategy": "resample",
}
CONDITIONAL_SLICE_FIGURE_STYLE = {
    "figure_dpi": 150,
    "cell_size": 2.35,
    "top_row_height": 2.5,
    "annotation_column_width": 3.2,
    "diagonal_kind": "kde",
    "diagonal_kde_points": 256,
    "diagonal_kde_bandwidth_scale": 1.0,
    "hist_alpha": 0.55,
    "hist_bins": 24,
    "unconstrained_color": "#797979",
    "unconstrained_scatter_alpha": 0.08,
    "unconstrained_scatter_size": 10.0,
    "conditional_scatter_alpha": 0.35,
    "conditional_scatter_size": 12.0,
    "slice_cmap": "tab10",
    "correlation_fontsize": 8.0,
    "correlation_loc": "upper left",
    "correlation_bbox_alpha": 0.75,
    "constrained_linewidth": 2.0,
    "constrained_label_fontsize": 9.0,
    "tick_labelsize": 8.0,
    "block_title_fontsize": 10.0,
    "block_text_fontsize": 8.5,
    "parameter_colors": PARAMETER_COLORS,
    "fallback_color": "#4C78A8",
    "max_free_params": 6,
}

RELEVANCE_GRID_ENABLED = True
RELEVANCE_K_NEAREST_PER_X0 = 100
RELEVANCE_GRID_FILENAME = "feature_parameter_relevance_grid.png"
RELEVANCE_GRID_STYLE = {
    "figure_dpi": 200,
    "cell_size": 3.5,
    "all_points_color": "#A0A0A0",
    "all_points_alpha": 0.25,
    "all_points_size": 20.0,
    "relevant_points_color": "#00A2FF",
    "relevant_points_alpha": 0.25,
    "relevant_points_size": 20.0,
    "text_fontsize": 6.0,
    "show_panel_annotations": True,
    "annotation_loc": "upper left",
    "show_grid": True,
    "parameter_colors": PARAMETER_COLORS,
    "feature_colors": FEATURES_COLORS,
    "show_experimental_guides": True,
    "experimental_guide_colors": EXPERIMENTAL_COLORS,
    "experimental_guide_cmap": "tab10",
    "experimental_guide_linewidth": 0.9,
    "experimental_guide_linestyle": "--",
    "experimental_guide_alpha": 0.78,
    "show_experimental_guide_legend": True,
    "experimental_guide_legend_fontsize": 8.0,
}

RELEVANCE_HEATMAP_ENABLED = True
RELEVANCE_HEATMAP_FILENAME = "feature_parameter_correlation_heatmaps.png"
RELEVANCE_HEATMAP_STYLE = {
    "figure_dpi": 200,
    "cell_size": 0.5,
    "cmap": "coolwarm",
    "vmin": -1.0,
    "vmax": 1.0,
    "show_cell_values": True,
    "text_fontsize": 6.0,
    "tick_labelsize": 6.0,
    "parameter_colors": PARAMETER_COLORS,
    "feature_colors": FEATURES_COLORS,
    "square_cells": True,
    "subplot_wspace": 0.45,
    "subplot_hspace": 0.55,
    "colorbar_shrink": 0.9,
    "panels": "all",  # "all", "relevant_pearson", "relevant_spearman", "all_pearson", "all_spearman"
}

# HELDOUT INTERNAL VALIDATION
# Trains N networks on random 90/10 splits of the simulated data and evaluates recovery on heldout simulated rows.
RUN_HELDOUT_INTERNAL_VALIDATION = True
HELDOUT_VALIDATION_N_NETWORKS = 5
HELDOUT_VALIDATION_TRAIN_FRACTION = 0.9
HELDOUT_VALIDATION_POSTERIOR_SAMPLES = 1000
HELDOUT_VALIDATION_RANDOM_SEED = 12345
HELDOUT_VALIDATION_OUTPUT_SUBDIR = "internal_validation_on_heldout_samples"
HELDOUT_VALIDATION_MASS_GRID = np.linspace(0.0, 1.0, 101)
HELDOUT_VALIDATION_MODE_GRID_POINTS = 256
HELDOUT_VALIDATION_MODE_BANDWIDTH_SCALE = 1.0
HELDOUT_VALIDATION_TRAIN_KWARGS = dict(SNPE_TRAIN_KWARGS)

HELDOUT_VALIDATION_ACCURACY_STYLE = {
    "figure_dpi": 140,
    "cell_width": 3.0,
    "cell_height": 2.7,
    "scatter_alpha": 0.1,
    "scatter_size": 20,
    "overlay_scatter_alpha": 0.05,
    "overlay_scatter_size": 8,
    "moving_average_window_fraction": 0.2,
    "moving_average_min_points": 10,
    "moving_average_stride_fraction": 0.25,
    "moving_average_lw": 2.0,
    "moving_average_alpha": 0.95,
    "overlay_curve_alpha": 1.0,
    "identity_color": "#7f7f7f",
    "identity_lw": 1.5,
    "identity_ls": "--",
    "scatter_color": "#007BFF",
    "overlay_color": "#007BFF",
    "text_fontsize": 8.0,
}

HELDOUT_VALIDATION_CALIBRATION_STYLE = {
    "figure_dpi": 140,
    "cell_width": 3.0,
    "cell_height": 2.7,
    "curve_lw": 2.0,
    "curve_alpha": 0.95,
    "overlay_curve_alpha": 0.25,
    "smoothing_window_points": 5,
    "identity_color": "#7f7f7f",
    "identity_lw": 1.5,
    "identity_ls": "--",
    "curve_color": "#D62728",
    "overlay_color": "#D62728",
    "text_fontsize": 8.0,
}


In [ ]:
# SELECTED PARAMETERS AND FEATURES
# Keep a few defaults uncommented so the notebook runs immediately.
# Use the template cell below to inspect all currently available parameters/features.

SELECTED_PARAMETERS = [
    # # # General
    # "SIM_PARAM_GENERAL_tau_size_um",

    # # # Baseline Input
    # "SIM_PARAM_BASELINE_INPUT_common_input_weight_sd",
    # "SIM_PARAM_BASELINE_INPUT_excitatory_input_baseline",
    # "SIM_PARAM_BASELINE_INPUT_lf_target_sd",
    # "SIM_PARAM_BASELINE_INPUT_alpha_target_sd_final",
    # "SIM_PARAM_BASELINE_INPUT_beta_target_sd_final",
    # "SIM_PARAM_BASELINE_INPUT_independent_input_power",
    # "SIM_PARAM_BASELINE_INPUT_independent_input_weight_sd",

    # Burst Input
    # "SIM_PARAM_BURST_INPUT_input_burst_sigma_ms",
    "SIM_PARAM_BURST_INPUT_lf_burst_peak_nA",
    "SIM_PARAM_BURST_INPUT_alpha_burst_peak_nA",
    "SIM_PARAM_BURST_INPUT_beta_burst_peak_nA",

    # Trend Input
    "SIM_PARAM_TREND_INPUT_lf_trend_slope_nA_per_s",
    "SIM_PARAM_TREND_INPUT_alpha_trend_slope_nA_per_s",
    "SIM_PARAM_TREND_INPUT_beta_trend_slope_nA_per_s",
]

SELECTED_FEATURES = [
    # # Firing Statistics
    # "OBS_FIRING_STATISTICS_firing_rate_mean",
    # "OBS_FIRING_STATISTICS_firing_rate_sd",
    # "OBS_FIRING_STATISTICS_firing_rate_skew",
    # "OBS_FIRING_STATISTICS_isi_cv_mean",
    # "OBS_FIRING_STATISTICS_isi_cv_sd",
    # "OBS_FIRING_STATISTICS_isi_cv_skew",

    # # Baseline
    # "OBS_BASELINE_CST_LF_raw_sd",
    # "OBS_BASELINE_ALPHA_raw_sd",
    # "OBS_BASELINE_BETA_raw_sd",
    # # "OBS_BASELINE_CST_LF_raw_mean",
    # # "OBS_BASELINE_ALPHA_raw_mean",
    # # "OBS_BASELINE_BETA_raw_mean",
    # # "OBS_BASELINE_ALPHA_mod_z_mean",
    # # "OBS_BASELINE_BETA_mod_z_mean",
    # "OBS_WINDOW_m3_to_m2_SYNC_mean",


    # Spectrum
    # "OBS_SPECTRUM_ALPHA_peak_hz",
    # "OBS_SPECTRUM_ALPHA_interval_low_hz",
    # "OBS_SPECTRUM_ALPHA_interval_high_hz",
    # "OBS_SPECTRUM_BETA_peak_hz",
    # "OBS_SPECTRUM_BETA_interval_low_hz",
    # "OBS_SPECTRUM_BETA_interval_high_hz",

    # Burst Features
    "OBS_BURST_FEATURES_resid_cst_low_frequency",
    "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_sync_coincidence_curve",

    # Trend Features
    "OBS_TREND_FEATURE_cst_low_frequency",
    "OBS_TREND_FEATURE_cst_alpha_modulation_hilbert_zscore",
    "OBS_TREND_FEATURE_cst_beta_modulation_hilbert_zscore",
    "OBS_TREND_FEATURE_sync_coincidence_curve",

    # Window Features ##########################
    #           # Use the template cell below for the full OBS_WINDOW_* list.
    # For baseline _______________________________
    # Baseline - Mean
    "OBS_WINDOW_m3_to_m2_CST_LF_raw_mean",
    "OBS_WINDOW_m3_to_m2_ALPHA_mod_z_mean",
    "OBS_WINDOW_m3_to_m2_BETA_mod_z_mean",
    "OBS_WINDOW_m3_to_m2_SYNC_mean",
    # Baseline - Min
    "OBS_WINDOW_m3_to_m2_CST_LF_raw_min",
    "OBS_WINDOW_m3_to_m2_ALPHA_mod_z_min",
    "OBS_WINDOW_m3_to_m2_BETA_mod_z_min",
    "OBS_WINDOW_m3_to_m2_SYNC_min",
    # Baseline - Max
    "OBS_WINDOW_m3_to_m2_CST_LF_raw_max",
    "OBS_WINDOW_m3_to_m2_ALPHA_mod_z_max",
    "OBS_WINDOW_m3_to_m2_BETA_mod_z_max",
    "OBS_WINDOW_m3_to_m2_SYNC_max",
    # For post-ready cue _________________________
    "OBS_WINDOW_m1_to_0_CST_LF_raw_mean",
    "OBS_WINDOW_m1_to_0_ALPHA_mod_z_mean",
    "OBS_WINDOW_m1_to_0_BETA_mod_z_mean",
    "OBS_WINDOW_m1_to_0_SYNC_mean",
    # Post-ready cue - Min
    "OBS_WINDOW_m1_to_0_CST_LF_raw_min",
    "OBS_WINDOW_m1_to_0_ALPHA_mod_z_min",
    "OBS_WINDOW_m1_to_0_BETA_mod_z_min",
    "OBS_WINDOW_m1_to_0_SYNC_min",
    # Post-ready cue - Max
    "OBS_WINDOW_m1_to_0_CST_LF_raw_max",
    "OBS_WINDOW_m1_to_0_ALPHA_mod_z_max",
    "OBS_WINDOW_m1_to_0_BETA_mod_z_max",
    "OBS_WINDOW_m1_to_0_SYNC_max",
    # For post-go/nogo cue  ______________________
    "OBS_WINDOW_0_to_1_CST_LF_raw_mean",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_mean",
    "OBS_WINDOW_0_to_1_BETA_mod_z_mean",
    "OBS_WINDOW_0_to_1_SYNC_mean",
    # Post-go/nogo cue - Min
    "OBS_WINDOW_0_to_1_CST_LF_raw_min",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_min",
    "OBS_WINDOW_0_to_1_BETA_mod_z_min",
    "OBS_WINDOW_0_to_1_SYNC_min",
    # Post-go/nogo cue - Max
    "OBS_WINDOW_0_to_1_CST_LF_raw_max",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_max",
    "OBS_WINDOW_0_to_1_BETA_mod_z_max",
    "OBS_WINDOW_0_to_1_SYNC_max",
    # Post-go/nogo cue - Timing of min/max
    "OBS_WINDOW_0_to_1_CST_LF_raw_t_max_rel_cue_s",
    "OBS_WINDOW_0_to_1_CST_LF_raw_t_min_rel_cue_s",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_t_max_rel_cue_s",
    "OBS_WINDOW_0_to_1_BETA_mod_z_t_max_rel_cue_s",
    "OBS_WINDOW_0_to_1_SYNC_t_max_rel_cue_s"
]


In [ ]:
# SCHEMA / TEMPLATE INSPECTION
sim_df_preview = load_summary_table(SIMULATED_TABLE_PATH)
exp_df_preview = load_summary_table(EXPERIMENTAL_TABLE_PATH)
templates = build_selection_templates(sim_df_preview, exp_df_preview)

display(Markdown("**Available simulated parameter groups**"))
display(pd.Series({k: len(v) for k, v in discover_available_parameter_groups(sim_df_preview).items()}, name="count"))
# display(Markdown("```python\n" + templates["parameters"] + "\n```"))

display(Markdown("**Available shared observation feature groups**"))
display(pd.Series({k: len(v) for k, v in discover_available_feature_groups(sim_df_preview, exp_df_preview).items()}, name="count"))
# display(Markdown("```python\n" + templates["features"] + "\n```"))


In [ ]:
# INPUT / OUTPUT PATHS
# Simulated table: one row = one simulated parameter-set batch summary.
# Experimental table: one row = one participant x session x condition batch summary.
# Use PKL by default because it preserves native objects better than CSV.
SIMULATED_TABLE_PATH = Path("sim_batches_GONOGO/parameter_set_summaries/parameter_set_feature_summary.pkl")
EXPERIMENTAL_TABLE_PATH = Path("EXP_DATA_ANALYZED_HDF5/_batch_summaries/experimental_batch_feature_summary.pkl")
OUTPUT_ROOT_DIR = Path("sbi_output_GONOGO")

# RUN MODE
# 'train' = prepare dataset from notebook selections, train a new SBI network, then sample posteriors.
# 'reload' = load an existing trained SBI bundle, ignore notebook parameter/feature selections,
#            prepare observations from EXPERIMENTAL_TABLE_PATH using the saved preprocessing,
#            then sample posteriors without retraining.
RUN_MODE = "train"
RELOAD_BUNDLE_PATH = OUTPUT_ROOT_DIR / "SBI_RUN_normalize_to_exp_FEATURES_all_windows" / "trained_inference_bundle.pkl"

# OBSERVATION LEVEL
# 'participant_condition' = one posterior per participant x session x condition row.
# 'condition_mean' = one posterior per condition mean row.
# 'grand_mean' = one posterior for the mean across all participant-condition rows.
OBSERVATION_LEVEL = "condition_mean"  # "participant_condition", "condition_mean", "grand_mean"

# FEATURE NORMALIZATION
# 'simulated' = z-score features using retained simulated training rows.
# 'experimental' = z-score features using experimental observations aggregated at FEATURE_NORMALIZATION_EXPERIMENTAL_LEVEL.
FEATURE_NORMALIZATION_SOURCE = "experimental"  # "simulated", "experimental"
FEATURE_NORMALIZATION_EXPERIMENTAL_LEVEL = "participant_condition"

# SBI / SAMPLING SETTINGS
DENSITY_ESTIMATOR = "maf"
TRAIN_DEVICE = "cpu"
SHOW_SBI_PROGRESS_BARS = True
N_TRAINED_NETWORKS = 5  # Equal-weight ensemble size; 1 preserves the historical single-network workflow.
ENSEMBLE_BASE_SEED = 12345  # Deterministically generates one training seed per network.
POSTERIOR_NUM_SAMPLES = 10_000 # 10_000  # Total pooled posterior samples across all networks.
POSTERIOR_SAMPLE_SEED = 12345

# Conservative defaults for a first pass. Increase later if needed.
SNPE_TRAIN_KWARGS = {
    "training_batch_size": 128, # 200,
    "learning_rate": 2e-4, # 5e-4,
    "validation_fraction": 0.1,
    "stop_after_epochs": 20,
    "show_train_summary": True,
}

# OPTIONAL MANUAL PARAMETER BOUND OVERRIDES
# Use exported parameter names as keys, e.g. "SIM_PARAM_GENERAL_tau_size_um": (0.0, 10.0)
# Leave empty to use bounds resolved automatically from *_priors_batch0.pkl.
PARAMETER_BOUNDS_OVERRIDES = {
    # "SIM_PARAM_GENERAL_tau_size_um": (0.0, 10.0),
}

# OUTPUT OPTIONS
SAVE_POSTERIOR_SAMPLES_PKL = True
SAVE_TRAINED_BUNDLE_PKL = True
SAVE_TRAINING_DIAGNOSTICS = True


In [ ]:
# TRAIN OR RELOAD SBI, THEN CREATE A NEW RUN OUTPUT SUBFOLDER
RUN_OUTPUT_DIR = create_run_output_dir(OUTPUT_ROOT_DIR, observation_level=OBSERVATION_LEVEL)
display(Markdown(f"**Run output directory**: `{RUN_OUTPUT_DIR}`"))

if RUN_MODE == "train":
    trained = train_posterior_ensemble(
        dataset,
        density_estimator=DENSITY_ESTIMATOR,
        device=TRAIN_DEVICE,
        show_progress_bars=SHOW_SBI_PROGRESS_BARS,
        train_kwargs=SNPE_TRAIN_KWARGS,
        n_networks=N_TRAINED_NETWORKS,
        ensemble_base_seed=ENSEMBLE_BASE_SEED,
    )
elif RUN_MODE == "reload":
    if loaded_trained is None:
        raise RuntimeError("Expected loaded_trained to be available in reload mode")
    trained = loaded_trained
else:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")


In [ ]:
# SAMPLE POSTERIORS, COMPUTE JOINT MODES, AND SAVE OUTPUTS
posterior_samples_df = sample_ensemble_posteriors_for_observations(
    trained,
    dataset,
    num_samples=POSTERIOR_NUM_SAMPLES,
    seed=POSTERIOR_SAMPLE_SEED,
)
posterior_modes_df = compute_posterior_modes(
    posterior_samples_df,
    parameter_names=dataset.selected_parameters,
    knn_neighbors=POSTERIOR_MODE_KNN_NEIGHBORS,
    min_neighbors=POSTERIOR_MODE_MIN_NEIGHBORS,
)
network_posterior_modes_df = compute_network_posterior_modes(
    posterior_samples_df,
    parameter_names=dataset.selected_parameters,
    knn_neighbors=POSTERIOR_MODE_KNN_NEIGHBORS,
    min_neighbors=POSTERIOR_MODE_MIN_NEIGHBORS,
)
posterior_summary_df = summarize_posterior_samples(
    posterior_samples_df,
    parameter_names=dataset.selected_parameters,
)
saved_paths = save_inference_outputs(
    output_dir=RUN_OUTPUT_DIR,
    dataset=dataset,
    trained=trained,
    posterior_samples_df=posterior_samples_df,
    posterior_summary_df=posterior_summary_df,
    posterior_modes_df=posterior_modes_df,
    network_posterior_modes_df=network_posterior_modes_df,
    save_posterior_samples_pkl=SAVE_POSTERIOR_SAMPLES_PKL,
    save_trained_bundle_pkl=SAVE_TRAINED_BUNDLE_PKL,
    save_training_diagnostics_figures=SAVE_TRAINING_DIAGNOSTICS,
)

display(Markdown("**Saved outputs**"))
display(pd.Series({k: str(v) for k, v in saved_paths.items()}, name="path"))
display(Markdown("**Posterior modes**"))
display(posterior_modes_df)
display(Markdown("**Posterior summary (first rows)**"))
display(posterior_summary_df.head(20))


In [ ]:
# POSTERIOR PAIRPLOTS
if PAIRPLOT_LABELS is None:
    if OBSERVATION_LEVEL == "participant_condition":
        labels_to_plot = [dataset.exp_observations.iloc[0]["observation_label"]]
    else:
        labels_to_plot = dataset.exp_observations["observation_label"].tolist()
else:
    labels_to_plot = list(PAIRPLOT_LABELS)

display(Markdown(f"**Posterior pairplots for labels**: {labels_to_plot}"))
pairplot_paths = plot_posterior_pairplots(
    posterior_samples_df,
    parameter_names=dataset.selected_parameters,
    parameter_bounds=dataset.parameter_bounds,
    posterior_modes_df=posterior_modes_df,
    network_posterior_modes_df=network_posterior_modes_df,
    selected_labels=labels_to_plot,
    output_dir=RUN_OUTPUT_DIR,
    **PAIRPLOT_STYLE,
)
display(pd.Series([str(path) for path in pairplot_paths], name="pairplot_path"))


## Posterior Condition Diagnostic


In [ ]:
# POSTERIOR CONDITION DIAGNOSTICS
posterior_condition_diagnostics_result = None

if POSTERIOR_CONDITION_DIAGNOSTICS_ENABLED:
    if POSTERIOR_CONDITION_DIAGNOSTICS_LABELS is None:
        condition_diagnostics_labels = dataset.exp_observations["observation_label"].tolist()
    else:
        condition_diagnostics_labels = list(POSTERIOR_CONDITION_DIAGNOSTICS_LABELS)

    display(Markdown(f"**Posterior condition diagnostics labels**: {condition_diagnostics_labels}"))
    posterior_condition_diagnostics_result = plot_posterior_condition_diagnostics(
        posterior_samples_df,
        parameter_names=dataset.selected_parameters,
        selected_labels=condition_diagnostics_labels,
        parameter_bounds=dataset.parameter_bounds,
        output_path=RUN_OUTPUT_DIR / POSTERIOR_CONDITION_DIAGNOSTICS_FILENAME,
        **POSTERIOR_CONDITION_DIAGNOSTICS_STYLE,
    )
    display(pd.Series({"output_path": str(posterior_condition_diagnostics_result["saved_path"])}, name="value"))
    display(Markdown("**Posterior probability above zero by condition**"))
    display(posterior_condition_diagnostics_result["p_gt_zero_df"])
    display(Markdown("**Posterior condition p_delta values**"))
    display(posterior_condition_diagnostics_result["p_delta_df"])
else:
    display(Markdown("**Posterior condition diagnostics disabled.**"))


## Posterior PCA


In [ ]:
# POSTERIOR PCA
posterior_pca_paths = []
posterior_pca_summaries = {}

if POSTERIOR_PCA_ENABLED:
    if POSTERIOR_PCA_LABELS is None:
        if OBSERVATION_LEVEL == "participant_condition":
            pca_labels_to_plot = [dataset.exp_observations.iloc[0]["observation_label"]]
        else:
            pca_labels_to_plot = dataset.exp_observations["observation_label"].tolist()
    else:
        pca_labels_to_plot = list(POSTERIOR_PCA_LABELS)

    display(Markdown(f"**Posterior PCA for labels**: {pca_labels_to_plot}"))
    posterior_pca_paths, posterior_pca_summaries = plot_posterior_pca_columns(
        posterior_samples_df,
        parameter_names=dataset.selected_parameters,
        posterior_modes_df=posterior_modes_df,
        selected_labels=pca_labels_to_plot,
        n_components_display=POSTERIOR_PCA_N_COMPONENTS_DISPLAY,
        output_dir=RUN_OUTPUT_DIR,
        filename_prefix=POSTERIOR_PCA_FILENAME_PREFIX,
        **POSTERIOR_PCA_STYLE,
    )
    display(pd.Series([str(path) for path in posterior_pca_paths], name="posterior_pca_path"))
else:
    display(Markdown("**Posterior PCA disabled.**"))


## Feature-Parameter Relevance Grid


In [ ]:
# FEATURE-PARAMETER RELEVANCE GRID
relevance_grid_metadata = None
relevance_grid_output_path = None
relevance_heatmap_metadata = None
relevance_heatmap_output_path = None

if RELEVANCE_GRID_ENABLED:
    relevance_grid_output_path = RUN_OUTPUT_DIR / RELEVANCE_GRID_FILENAME
    _, _, relevance_grid_metadata = plot_feature_parameter_relevance_grid(
        dataset,
        posterior_modes_df=posterior_modes_df,
        k_nearest_per_x0=RELEVANCE_K_NEAREST_PER_X0,
        output_path=relevance_grid_output_path,
        **RELEVANCE_GRID_STYLE,
    )
    display(Markdown("**Feature-parameter relevance grid**"))
    display(pd.Series({
        "output_path": str(relevance_grid_output_path),
        "normalization": relevance_grid_metadata["normalization"],
        "k_requested": relevance_grid_metadata["k_requested"],
        "k_effective": relevance_grid_metadata["k_effective"],
        "n_observations": relevance_grid_metadata["n_observations"],
        "n_relevant_simulations": relevance_grid_metadata["n_relevant_simulations"],
        "n_simulations": relevance_grid_metadata["n_simulations"],
        "fraction_relevant": relevance_grid_metadata["fraction_relevant"],
    }, name="value"))
else:
    display(Markdown("**Feature-parameter relevance grid disabled.**"))

if RELEVANCE_HEATMAP_ENABLED:
    relevance_heatmap_output_path = RUN_OUTPUT_DIR / RELEVANCE_HEATMAP_FILENAME
    _, _, relevance_heatmap_metadata = plot_feature_parameter_correlation_heatmaps(
        dataset,
        k_nearest_per_x0=RELEVANCE_K_NEAREST_PER_X0,
        output_path=relevance_heatmap_output_path,
        **RELEVANCE_HEATMAP_STYLE,
    )
    display(Markdown("**Feature-parameter correlation heatmaps**"))
    display(pd.Series({
        "output_path": str(relevance_heatmap_output_path),
        "normalization": relevance_heatmap_metadata["normalization"],
        "k_requested": relevance_heatmap_metadata["k_requested"],
        "k_effective": relevance_heatmap_metadata["k_effective"],
        "n_observations": relevance_heatmap_metadata["n_observations"],
        "n_relevant_simulations": relevance_heatmap_metadata["n_relevant_simulations"],
        "n_simulations": relevance_heatmap_metadata["n_simulations"],
        "fraction_relevant": relevance_heatmap_metadata["fraction_relevant"],
        "panels_displayed": ", ".join(relevance_heatmap_metadata["panels_displayed"]),
    }, name="value"))
else:
    display(Markdown("**Feature-parameter correlation heatmaps disabled.**"))


## Conditional Posterior Slice Explorer


In [ ]:
# CONDITIONAL POSTERIOR SLICE EXPLORER
conditional_slice_paths = []
conditional_slice_results = {}
conditional_slice_metadata_tables = []

if CONDITIONAL_SLICE_EXPLORER_ENABLED:
    if not CONDITIONAL_SLICE_CONSTRAINED_PARAMS:
        raise ValueError("Set CONDITIONAL_SLICE_CONSTRAINED_PARAMS before enabling the conditional slice explorer")

    if CONDITIONAL_SLICE_LABELS is None:
        if OBSERVATION_LEVEL == "participant_condition":
            conditional_slice_labels = [dataset.exp_observations.iloc[0]["observation_label"]]
        else:
            conditional_slice_labels = dataset.exp_observations["observation_label"].tolist()
    else:
        conditional_slice_labels = list(CONDITIONAL_SLICE_LABELS)

    display(Markdown(f"**Conditional posterior slices for labels**: {conditional_slice_labels}"))
    for label in conditional_slice_labels:
        safe_label = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(label))
        output_path = RUN_OUTPUT_DIR / f"conditional_posterior_slices__{safe_label}.png"
        result = plot_conditional_posterior_slices(
            trained,
            dataset,
            posterior_samples_df,
            posterior_modes_df=posterior_modes_df,
            parameter_names=dataset.selected_parameters,
            observation_label=str(label),
            constrained_param_names=CONDITIONAL_SLICE_CONSTRAINED_PARAMS,
            slice_percentiles=CONDITIONAL_SLICE_PERCENTILES,
            include_mode_slice=CONDITIONAL_SLICE_INCLUDE_MODE,
            reference_posterior_samples=CONDITIONAL_SLICE_REFERENCE_SAMPLES,
            n_conditional_samples=CONDITIONAL_SLICE_N_CONDITIONAL_SAMPLES,
            random_seed=CONDITIONAL_SLICE_RANDOM_SEED,
            mismatch_warn_threshold=CONDITIONAL_SLICE_MISMATCH_WARN_THRESHOLD,
            output_path=output_path,
            **CONDITIONAL_SLICE_MCMC_STYLE,
            **CONDITIONAL_SLICE_FIGURE_STYLE,
        )
        conditional_slice_results[str(label)] = result
        if result["saved_path"] is not None:
            conditional_slice_paths.append(result["saved_path"])
        slice_meta = result["slice_metadata_df"].copy()
        slice_meta.insert(0, "plot_path", str(output_path))
        conditional_slice_metadata_tables.append(slice_meta)

    if conditional_slice_paths:
        display(pd.Series([str(path) for path in conditional_slice_paths], name="conditional_slice_path"))
    if conditional_slice_metadata_tables:
        display(Markdown("**Conditional slice metadata**"))
        display(pd.concat(conditional_slice_metadata_tables, ignore_index=True))
else:
    display(Markdown("**Conditional posterior slice explorer disabled.**"))


## Internal Validation On Heldout Simulated Data
Cross-validation-style diagnostics for parameter recovery and calibration.


In [ ]:
# RUN HELDOUT INTERNAL VALIDATION
validation_results = None
validation_saved_paths = {}
validation_output_dir = None
validation_base_dataset = dataset if RUN_MODE == "train" else loaded_training_dataset

if RUN_HELDOUT_INTERNAL_VALIDATION:
    if validation_base_dataset is None:
        raise RuntimeError("No training dataset available for heldout validation.")
    validation_output_dir = RUN_OUTPUT_DIR / HELDOUT_VALIDATION_OUTPUT_SUBDIR
    validation_results = run_internal_posterior_validation(
        validation_base_dataset,
        n_networks=HELDOUT_VALIDATION_N_NETWORKS,
        train_fraction=HELDOUT_VALIDATION_TRAIN_FRACTION,
        posterior_num_samples=HELDOUT_VALIDATION_POSTERIOR_SAMPLES,
        random_seed=HELDOUT_VALIDATION_RANDOM_SEED,
        density_estimator=DENSITY_ESTIMATOR,
        device=TRAIN_DEVICE,
        show_progress_bars=SHOW_SBI_PROGRESS_BARS,
        train_kwargs=HELDOUT_VALIDATION_TRAIN_KWARGS,
        mass_grid=HELDOUT_VALIDATION_MASS_GRID,
        mode_grid_points=HELDOUT_VALIDATION_MODE_GRID_POINTS,
        mode_bandwidth_scale=HELDOUT_VALIDATION_MODE_BANDWIDTH_SCALE,
    )
    validation_saved_paths = save_internal_validation_outputs(
        output_dir=validation_output_dir,
        validation_results=validation_results,
    )
    display(Markdown(f"**Heldout validation output directory**: `{validation_output_dir}`"))
    display(pd.Series({k: str(v) for k, v in validation_saved_paths.items()}, name="path"))
    display(Markdown("**Heldout validation config**"))
    display(pd.Series(validation_results["config"]))
else:
    display(Markdown("**Heldout internal validation disabled.**"))


In [ ]:
# HELDOUT VALIDATION ACCURACY PLOT
if RUN_HELDOUT_INTERNAL_VALIDATION:
    accuracy_validation_path = plot_internal_validation_accuracy(
        validation_results,
        parameter_bounds=validation_base_dataset.parameter_bounds,
        parameter_names=validation_base_dataset.selected_parameters,
        output_dir=validation_output_dir,
        **HELDOUT_VALIDATION_ACCURACY_STYLE,
    )
    display(pd.Series([str(accuracy_validation_path)], name="heldout_accuracy_figure"))
else:
    display(Markdown("**Heldout internal validation disabled.**"))


In [ ]:
# HELDOUT VALIDATION CALIBRATION PLOT
if RUN_HELDOUT_INTERNAL_VALIDATION:
    calibration_validation_path = plot_internal_validation_calibration(
        validation_results,
        parameter_names=validation_base_dataset.selected_parameters,
        output_dir=validation_output_dir,
        **HELDOUT_VALIDATION_CALIBRATION_STYLE,
    )
    display(pd.Series([str(calibration_validation_path)], name="heldout_calibration_figure"))
else:
    display(Markdown("**Heldout internal validation disabled.**"))
